# Notebook 1.5: Thermal States and Mixed Radiation Fields
### Module 1: Single-Mode Quantum States of Light
**Computational Quantum Optics Curriculum**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_1_states_of_light/06_thermal_and_mixed_states.ipynb)

---

### Prerequisites
- Notebook 1.1: Fock States, Ladder Operators, and Truncation Artifacts
- Notebook 1.2: Coherent States, Displacement, and Poissonian Statistics
- Notebook 1.3: Squeezed Vacuum and Quadrature Noise Reduction
- Notebook 1.4: Displaced Squeezed States and Phase Space Interference

### Pedagogical Learning Objectives
1. **Density Matrix Formalism for Statistical Mixtures:** Formulate the canonical equilibrium thermal state $\hat{\rho}_{\text{th}} = e^{-\beta \hbar \omega \hat{a}^\dagger \hat{a}} / \mathcal{Z}$ in discrete Fock basis and express it in terms of the mean photon occupancy $\bar{n}$.
2. **Bose-Einstein vs. Poissonian Statistics:** Contrast the geometric Bose-Einstein distribution $P_n = \bar{n}^n / (1+\bar{n})^{n+1}$ with Poissonian coherent light, proving why $n=0$ is unconditionally the most probable photon number.
3. **Photon Fluctuation Partitioning:** Decompose the thermal variance $\langle (\Delta \hat{n})^2 \rangle = \bar{n} + \bar{n}^2$ into discrete particle noise ($\bar{n}$) and wave interference / intensity fluctuation noise ($\bar{n}^2$).
4. **Purity and von Neumann Entropy:** Quantify mixedness via purity $\mathcal{P} = \mathrm{Tr}(\hat{\rho}^2) = 1/(2\bar{n}+1)$ and von Neumann entropy $S(\hat{\rho}) = (\bar{n}+1)\ln(\bar{n}+1) - \bar{n}\ln\bar{n}$.
5. **Phase-Diffused Coherent States:** Compare true thermal blackbody radiation with a phase-randomized laser beam, demonstrating how loss of phase coherence differs from thermalization.
6. **Numerical Benchmarking:** Implement Fock-space density matrix solvers, verify normalization and positive semidefiniteness, and compare against QuTiP `thermal_dm()`.


In [ ]:
# Setup, environment detection, and library imports
import os
import sys
import math
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

# Ensure matplotlib handles headless environments gracefully
if 'MPLBACKEND' not in os.environ:
    os.environ['MPLBACKEND'] = 'Agg' if not hasattr(sys, 'ps1') else 'module://matplotlib_inline.backend_inline'

# Plot styling configuration for high-DPI publication graphics
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['grid.alpha'] = 0.35

# Detect QuTiP availability with clean fallback
try:
    import qutip as qt
    HAS_QUTIP = True
    print(f"QuTiP {qt.__version__} successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

# Detect interactive widgets environment
try:
    from ipywidgets import interact, FloatSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


## 1. Density Matrix Formalism of Thermal Radiation

Unlike pure states ($|\psi\rangle$), real-world optical fields in thermal equilibrium with a heat reservoir at temperature $T$ cannot be described by a single state vector. Instead, they constitute an **incoherent statistical ensemble** described by a density operator $\hat{\rho}$.

According to equilibrium statistical mechanics, the canonical density operator for a single cavity mode of frequency $\omega$ is:

$$
\hat{\rho}_{\text{th}} = \frac{1}{\mathcal{Z}} \exp\left(-\beta \hbar \omega \hat{a}^\dagger \hat{a}\right) = \frac{1}{\mathcal{Z}} \sum_{n=0}^\infty e^{-\beta \hbar \omega n} |n\rangle\langle n|
$$

where $\beta = \frac{1}{k_B T}$ and $\mathcal{Z}$ is the partition function:

$$
\mathcal{Z} = \mathrm{Tr}\left[e^{-\beta \hbar \omega \hat{a}^\dagger \hat{a}}\right] = \sum_{n=0}^\infty e^{-\beta \hbar \omega n} = \frac{1}{1 - e^{-\beta \hbar \omega}}
$$

---

### The Bose-Einstein Distribution in Fock Space

The mean thermal photon occupancy $\bar{n}$ is given by the Planck formula:

$$
\bar{n} = \langle \hat{a}^\dagger \hat{a} \rangle = \frac{1}{e^{\beta \hbar \omega} - 1} \quad \Longleftrightarrow \quad e^{-\beta \hbar \omega} = \frac{\bar{n}}{\bar{n} + 1}
$$

Substituting this parameterization back into $\hat{\rho}_{\text{th}}$ eliminates temperature and frequency in favor of the single observable parameter $\bar{n}$:

$$
\hat{\rho}_{\text{th}} = \sum_{n=0}^\infty P_n |n\rangle\langle n|, \qquad P_n = \frac{\bar{n}^n}{(1 + \bar{n})^{n+1}}
$$

This is the famous **geometric (Bose-Einstein) distribution**.

---

### Key Observations:
1. **The Vacuum is King ($P_0 > P_1 > P_2 \dots$):**
   Notice that $P_0 = \frac{1}{1 + \bar{n}}$. Since $\frac{\bar{n}}{1+\bar{n}} < 1$, $P_n$ decreases monotonically with $n$!
   **Zero photons is always the single most probable measurement result**, even when the average photon number is huge ($\bar{n} = 1000$)!
2. **Diagonal in Fock Basis:**
   Because all off-diagonal coherences $\langle n | \hat{\rho}_{\text{th}} | m \rangle = 0$ ($n \neq m$) vanish, thermal radiation contains zero optical phase coherence.


In [ ]:
# Numerical Construction and Diagnostic Functions for Thermal Density Matrices

def thermal_density_matrix(n_bar, N):
    """Constructs truncated Fock-space density matrix rho for thermal state with mean n_bar."""
    if n_bar == 0:
        rho = np.zeros((N, N), dtype=np.complex128)
        rho[0, 0] = 1.0
        return rho, 0.0, 0.0
    
    # Calculate unnormalized probabilities P_n = [n_bar / (1 + n_bar)]^n
    ratio = n_bar / (1.0 + n_bar)
    n_vals = np.arange(N, dtype=np.float64)
    p_n = (1.0 / (1.0 + n_bar)) * (ratio ** n_vals)
    
    # Diagonal density matrix
    rho = np.diag(p_n).astype(np.complex128)
    
    # Diagnostics: trace loss and tail truncation error
    tr = np.real(np.trace(rho))
    trace_loss = abs(1.0 - tr)
    tail_error = p_n[-1]
    
    # Renormalize to ensure exact unit trace in truncated basis
    rho = rho / tr
    return rho, trace_loss, tail_error

def state_purity(rho):
    """Calculates purity P = Tr(rho^2)."""
    return np.real(np.trace(rho @ rho))

def von_neumann_entropy(rho, base=np.e):
    """Calculates von Neumann entropy S(rho) = -Tr(rho ln rho)."""
    # Diagonalize rho to find eigenvalues (probabilities)
    evals = la.eigvalsh(rho)
    # Filter strictly positive eigenvalues to avoid log(0)
    pos_evals = evals[evals > 1e-15]
    if base == 2:
        return -np.sum(pos_evals * np.log2(pos_evals))
    return -np.sum(pos_evals * np.log(pos_evals))

# Verification against exact analytical formulas
n_bar_test = 2.5
N_test = 50

rho_num, tr_err, tail = thermal_density_matrix(n_bar_test, N_test)
purity_num = state_purity(rho_num)
entropy_num = von_neumann_entropy(rho_num)

# Analytical exact expressions
purity_ana = 1.0 / (2.0 * n_bar_test + 1.0)
entropy_ana = (n_bar_test + 1.0) * np.log(n_bar_test + 1.0) - n_bar_test * np.log(n_bar_test)

print(f"--- Thermal State Diagnostics (n_bar = {n_bar_test}, Cutoff N = {N_test}) ---")
print(f"Trace Error:              {tr_err:.2e}")
print(f"Tail Truncation Error:    {tail:.2e}")
print(f"Numerical Purity:         {purity_num:.8f} | Analytical: {purity_ana:.8f}")
print(f"Numerical Entropy (nats): {entropy_num:.8f} | Analytical: {entropy_ana:.8f}")

assert abs(purity_num - purity_ana) < 1e-6, "Purity numerical benchmark failed!"
assert abs(entropy_num - entropy_ana) < 1e-5, "Entropy numerical benchmark failed!"
print("Numerical implementation perfectly verified against analytical benchmarks!")


## 2. Fluctuation Partitioning: Particle vs. Wave Noise

Let us calculate the photon number variance of a thermal state. Using the geometric progression relations:

$$
\langle \hat{n} \rangle = \sum_{n=0}^\infty n P_n = \bar{n}
$$
$$
\langle \hat{n}^2 \rangle = \sum_{n=0}^\infty n^2 P_n = \bar{n} + 2\bar{n}^2
$$

Therefore, the variance is:

$$
\langle (\Delta \hat{n})^2 \rangle = \langle \hat{n}^2 \rangle - \langle \hat{n} \rangle^2 = \bar{n} + \bar{n}^2 = \bar{n}(1 + \bar{n})
$$

This remarkable formula, originally deduced by **Albert Einstein in 1909** for blackbody radiation, provided early historical evidence for wave-particle duality:

$$
\underbrace{\langle (\Delta \hat{n})^2 \rangle}_{\text{Total Noise}} = \underbrace{\bar{n}}_{\text{Particle (Shot) Noise}} + \underbrace{\bar{n}^2}_{\text{Wave (Intensity) Noise}}
$$

1. **Particle Term ($\bar{n}$):** Arises from the discrete nature of photons arriving randomly at a detector (identical to Poissonian shot noise of a classical particle beam).
2. **Wave Term ($\bar{n}^2$):** Arises from the interference and beat fluctuations between different random classical electromagnetic wave modes (the Hanbury Brown and Twiss effect / photon bunching).

---

### Comparison of Photon Number Statistics Across States

| Quantity | Coherent State $|\alpha\rangle$ | Thermal State $\hat{\rho}_{\text{th}}$ | Squeezed Vacuum $|0, r\rangle$ |
| :--- | :--- | :--- | :--- |
| **Distribution $P_n$** | Poissonian: $e^{-\bar{n}} \frac{\bar{n}^n}{n!}$ | Bose-Einstein: $\frac{\bar{n}^n}{(1+\bar{n})^{n+1}}$ | Even Fock: $\frac{(2m)!}{2^{2m}(m!)^2} \frac{(\tanh r)^{2m}}{\cosh r}$ |
| **Mean $\langle n \rangle$** | $|\alpha|^2$ | $\bar{n}$ | $\sinh^2 r$ |
| **Variance $\text{Var}(n)$** | $\bar{n}$ | $\bar{n} + \bar{n}^2$ | $2\bar{n}(\bar{n}+1)$ |
| **Relative Noise $\frac{\Delta n}{\langle n \rangle}$** | $\frac{1}{\sqrt{\bar{n}}} \to 0$ (deterministic) | $\sqrt{1 + \frac{1}{\bar{n}}} \to 1$ (100% noise) | $\sqrt{2 + \frac{2}{\bar{n}}} \to \sqrt{2}$ |
| **Mandel $Q$ Parameter** | $0$ (Poissonian) | $+\bar{n} > 0$ (Super-Poissonian) | $1 + 2\bar{n} > 0$ (Super-Poissonian) |


In [ ]:
# Comparative Simulation: Bose-Einstein vs. Poissonian vs. Phase-Diffused Light

n_bar = 4.0
N_dim = 30
n_axis = np.arange(N_dim)

# 1. Thermal State (Bose-Einstein Distribution)
rho_th, _, _ = thermal_density_matrix(n_bar, N_dim)
p_thermal = np.real(np.diag(rho_th))

# 2. Coherent State (Poissonian Distribution with identical mean)
p_coherent = np.array([(n_bar**n) * np.exp(-n_bar) / math.factorial(n) for n in n_axis])

# 3. Phase-Diffused Laser (Coherent state with completely randomized phase)
# Note: Has identical Poissonian photon statistics P_n, but zero off-diagonal coherence!
p_phasediff = p_coherent.copy()

# Plotting: Multi-panel comparative analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Probability Distribution P(n)
width = 0.38
ax1.bar(n_axis - width/2, p_thermal, width=width, color='#d62728', alpha=0.8,
        label=rf'Thermal (Bose-Einstein, $\bar{{n}}={n_bar}$): $P_0={p_thermal[0]:.2f}$')
ax1.bar(n_axis + width/2, p_coherent, width=width, color='#1f77b4', alpha=0.8,
        label=rf'Coherent (Poissonian, $|\alpha|^2={n_bar}$): $P_0={p_coherent[0]:.2f}$')

ax1.axvline(n_bar, color='black', ls='--', lw=1.8, label=rf'Mean $\langle n \rangle = {n_bar}$')
ax1.set_xlim(-0.6, 22)
ax1.set_xlabel(r"Photon Number $n$")
ax1.set_ylabel(r"Probability $P(n)$")
ax1.set_title("Photon Counting Statistics: Monotonic Thermal vs. Peaked Coherent")
ax1.legend(loc='upper right', framealpha=0.9)
ax1.grid(True)

# Panel 2: Density Matrix Heatmaps (Fock Representation)
# Show the first 8x8 block of the thermal density matrix
n_sub = 8
im = ax2.imshow(np.real(rho_th[:n_sub, :n_sub]), cmap='viridis', interpolation='nearest',
                extent=[-0.5, n_sub-0.5, n_sub-0.5, -0.5])
plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04, label=r'$\mathrm{Re}(\rho_{n,m})$')

# Annotate values on heatmap
for i in range(n_sub):
    for j in range(n_sub):
        val = np.real(rho_th[i, j])
        if val > 0.01:
            ax2.text(j, i, f"{val:.2f}", ha='center', va='center',
                     color='white' if val < 0.15 else 'black', fontsize=8)

ax2.set_xticks(range(n_sub))
ax2.set_yticks(range(n_sub))
ax2.set_xlabel(r"Column index $m$")
ax2.set_ylabel(r"Row index $n$")
ax2.set_title(r"Thermal State $\hat{\rho}_{\mathrm{th}}$ in Fock Basis (Strictly Diagonal)")

plt.tight_layout()
plt.show()


## 3. Purity, Entropy, and the Thermalization of Coherent Light

### Purity Analysis

The **purity** of a quantum state $\hat{\rho}$ measures how close it is to a single pure state:

$$
\mathcal{P} = \mathrm{Tr}(\hat{\rho}^2) = \sum_{n=0}^\infty P_n^2
$$

For thermal light:

$$
\mathcal{P}_{\text{th}} = \sum_{n=0}^\infty \left[ \frac{\bar{n}^n}{(1+\bar{n})^{n+1}} \right]^2 = \frac{1}{(1+\bar{n})^2} \sum_{n=0}^\infty \left(\frac{\bar{n}}{1+\bar{n}}\right)^{2n} = \frac{1}{(1+\bar{n})^2} \frac{1}{1 - \frac{\bar{n}^2}{(1+\bar{n})^2}} = \frac{1}{2\bar{n} + 1}
$$

- At $T = 0$ ($\bar{n} = 0$): $\mathcal{P} = 1$ (the vacuum state $|0\rangle$, which is pure).
- In the macroscopic thermal limit ($\bar{n} \gg 1$): $\mathcal{P} \approx \frac{1}{2\bar{n}} \to 0$ (maximally mixed over infinite Fock states).

---

### von Neumann Entropy

The **von Neumann entropy** quantifies the quantum statistical uncertainty or missing information in nats (using natural log) or bits (using base 2):

$$
S(\hat{\rho}) = - \mathrm{Tr}(\hat{\rho} \ln \hat{\rho}) = - \sum_{n=0}^\infty P_n \ln P_n = (\bar{n}+1)\ln(\bar{n}+1) - \bar{n}\ln\bar{n}
$$

As $\bar{n} \to 0$, $S(\hat{\rho}) \to 0$. As $\bar{n} \gg 1$, $S(\hat{\rho}) \approx \ln\bar{n} + 1$.

---

### Thermal Light vs. Phase-Diffused Laser: A Critical Distinction

What is the difference between a **thermal state** and a **laser beam that has lost its phase coherence**?

1. **Phase-Diffused Coherent State:**
   If a coherent state $|\alpha\rangle$ undergoes complete random phase diffusion over $\phi \in [0, 2\pi)$:
   $$
   \hat{\rho}_{\text{pd}} = \frac{1}{2\pi} \int_0^{2\pi} |\alpha e^{i\phi}\rangle\langle \alpha e^{i\phi}| d\phi = \sum_{n=0}^\infty e^{-|\alpha|^2} \frac{|\alpha|^{2n}}{n!} |n\rangle\langle n|
   $$
   This state is diagonal, but its photon distribution is **Poissonian**! Its purity is:
   $$
   \mathcal{P}_{\text{pd}} = \sum_{n=0}^\infty P_n^2 = e^{-2\bar{n}} I_0(2\bar{n}) \approx \frac{1}{\sqrt{4\pi \bar{n}}}
   $$
   where $I_0$ is the modified Bessel function.

2. **Thermal State:**
   A thermal state is an incoherent Gaussian mixture of coherent states over **both amplitude and phase**:
   $$
   \hat{\rho}_{\text{th}} = \int \frac{e^{-|\alpha|^2/\bar{n}}}{\pi \bar{n}} |\alpha\rangle\langle \alpha| d^2\alpha
   $$
   Because amplitude also fluctuates randomly according to Rayleigh-Jeans distribution, its purity drops much faster ($\mathcal{P}_{\text{th}} \approx \frac{1}{2\bar{n}}$ vs. $\mathcal{P}_{\text{pd}} \approx \frac{1}{\sqrt{4\pi\bar{n}}}$).


In [ ]:
# Scaling Analysis: Purity and von Neumann Entropy as Functions of Mean Occupancy

n_bar_sweep = np.linspace(0.01, 10.0, 100)

# 1. Thermal State Scaling
purity_th_exact = 1.0 / (2.0 * n_bar_sweep + 1.0)
entropy_th_exact = (n_bar_sweep + 1.0) * np.log(n_bar_sweep + 1.0) - n_bar_sweep * np.log(n_bar_sweep)

# 2. Phase-Diffused Laser Purity Scaling
import scipy.special as sp
purity_pd_exact = np.exp(-2.0 * n_bar_sweep) * sp.i0(2.0 * n_bar_sweep)

# Plotting: Two-panel scaling curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: State Purity Tr(rho^2)
ax1.plot(n_bar_sweep, purity_th_exact, 'r-', lw=2.5, label=r'Thermal State: $\mathcal{P} = 1/(2\bar{n}+1)$')
ax1.plot(n_bar_sweep, purity_pd_exact, 'b--', lw=2.2, label=r'Phase-Diffused Laser: $\mathcal{P} = e^{-2\bar{n}} I_0(2\bar{n})$')
ax1.axhline(1.0, color='gray', ls=':', label=r'Pure State Bound ($\mathcal{P}=1$)')

ax1.set_xlim(0, 10)
ax1.set_ylim(0, 1.05)
ax1.set_xlabel(r"Mean Photon Occupancy $\bar{n}$")
ax1.set_ylabel(r"Purity $\mathcal{P} = \mathrm{Tr}(\hat{\rho}^2)$")
ax1.set_title("Quantum State Purity: Thermal vs. Phase-Diffused Light")
ax1.legend(loc='upper right', framealpha=0.9)
ax1.grid(True)

# Panel 2: von Neumann Entropy S(rho)
ax2.plot(n_bar_sweep, entropy_th_exact, color='darkgreen', lw=2.5,
         label=r'Thermal State Entropy $S(\hat{\rho})$')
ax2.plot(n_bar_sweep, np.log(n_bar_sweep) + 1.0, 'k:', lw=1.8, label=r'Asymptotic Limit: $\ln(\bar{n}) + 1$')

ax2.set_xlim(0, 10)
ax2.set_ylim(0, 3.5)
ax2.set_xlabel(r"Mean Photon Occupancy $\bar{n}$")
ax2.set_ylabel(r"von Neumann Entropy $S(\hat{\rho})$ [nats]")
ax2.set_title("Thermal Information Loss: von Neumann Entropy Growth")
ax2.legend(loc='lower right', framealpha=0.9)
ax2.grid(True)

plt.tight_layout()
plt.show()


> **Physical Insight: Why Does Thermal Light Exhibit Photon Bunching?**
> A thermal state exhibits super-Poissonian fluctuations $\langle (\Delta n)^2 \rangle = \bar{n} + \bar{n}^2 > \bar{n}$, which is the statistical signature of **photon bunching** ($g^{(2)}(0) = 2$).
>
> Physically, photons are bosons described by symmetric wavefunctions. In a thermal beam, multiple independent radiative emissions interfere randomly. When wave amplitudes add constructively, the local intensity surges, increasing the conditional probability of detecting a second photon immediately after a first photon! This is fundamentally classical wave interference combined with Bose-Einstein counting statistics.


In [ ]:
# Interactive Exploration: Thermal Equilibrium & Blackbody Photon Statistics

def explore_thermal_state(n_bar=2.0, N_cutoff=30):
    """Interactive visualization of thermal state statistics, purity, and entropy."""
    n_arr = np.arange(N_cutoff)
    rho_th, _, _ = thermal_density_matrix(n_bar, N_cutoff)
    p_th = np.real(np.diag(rho_th))
    
    # Coherent state for comparison
    p_coh = np.array([(n_bar**n) * np.exp(-n_bar) / math.factorial(n) for n in n_arr])
    
    purity = state_purity(rho_th)
    entropy = von_neumann_entropy(rho_th)
    var_n = n_bar * (1.0 + n_bar)
    mandel_q = n_bar
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Panel 1: Probability distributions
    ax1.plot(n_arr, p_th, 'ro-', lw=2, markersize=5, label=rf'Thermal: $P_0={p_th[0]:.3f}$')
    ax1.plot(n_arr, p_coh, 'bs--', lw=1.8, markersize=5, label=rf'Coherent: $P_0={p_coh[0]:.3f}$')
    ax1.axvline(n_bar, color='black', ls=':', label=rf'$\langle n \rangle = {n_bar:.1f}$')
    
    ax1.set_xlim(-0.5, min(N_cutoff-1, int(3*n_bar + 5)))
    ax1.set_xlabel("Photon Number n")
    ax1.set_ylabel("Probability P(n)")
    ax1.set_title(rf"Photon Statistics ($\bar{{n}} = {n_bar:.2f}$)")
    ax1.legend(loc='upper right')
    ax1.grid(True)
    
    # Panel 2: Summary Metrics Bar Chart
    metrics = [purity, entropy, var_n, mandel_q]
    labels = [r'Purity $\mathcal{P}$', r'Entropy $S$', r'Var$(n)$', r'Mandel $Q$']
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    bars = ax2.bar(labels, metrics, color=colors, alpha=0.8, edgecolor='black', width=0.55)
    for bar, val in zip(bars, metrics):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f"{val:.2f}", ha='center', va='bottom', fontweight='bold')
        
    ax2.set_ylim(0, max(metrics) * 1.25 + 0.5)
    ax2.set_title("Thermal Statistical Indicators")
    ax2.grid(True, axis='y')
    
    plt.tight_layout()
    plt.show()

# Launch interactive exploration
if HAS_WIDGETS:
    interact(explore_thermal_state,
             n_bar=FloatSlider(min=0.1, max=8.0, step=0.2, value=2.0, description=r'Mean \bar{n}:'),
             N_cutoff=FloatSlider(min=15, max=50, step=5, value=30, description=r'Cutoff N:'))
else:
    # Static execution fallback for headless environments
    explore_thermal_state(n_bar=2.0, N_cutoff=30)


## 4. Self-Check Exercises and Pedagogical Challenges

Deepen your comprehension of thermal radiation and mixed states through these analytical and numerical challenges.

---

### Challenge 1: The Cosmic Microwave Background (CMB) at Optical vs. Microwave Frequencies
The CMB blackbody radiation has a temperature $T \approx 2.725\text{ K}$.
1. Calculate the mean thermal photon number $\bar{n} = \frac{1}{e^{\hbar\omega/k_BT} - 1}$ for:
   - Microwave frequency $\nu = 100\text{ GHz}$ (peak CMB wavelength region).
   - Optical frequency $\nu = 500\text{ THz}$ (visible red/green light, $\lambda \approx 600\text{ nm}$).
2. Explain why visible optical fields at room temperature ($T = 300\text{ K}$) have virtually zero thermal photon background ($\bar{n} \ll 10^{-10}$), whereas microwave circuits require dilution refrigerators ($T \sim 10\text{ mK}$) to avoid thermal noise.

<details>
<summary><b>Click to expand Solution to Challenge 1</b></summary>

**Part 1 Solution:**
Recall $k_B \approx 1.3806 \times 10^{-23}\text{ J/K}$ and $h \approx 6.626 \times 10^{-34}\text{ J}\cdot\text{s}$.
The thermal energy scale at $T = 2.725\text{ K}$ is $k_BT \approx 3.76 \times 10^{-23}\text{ J} \approx 0.235\text{ meV}$.

- For $\nu = 100\text{ GHz}$:
  Photon energy $h\nu = 6.626 \times 10^{-23}\text{ J}$.
  Ratio $x = \frac{h\nu}{k_BT} = \frac{6.626 \times 10^{-23}}{3.76 \times 10^{-23}} \approx 1.762$.
  Mean photon number:
  $$
  \bar{n} = \frac{1}{e^{1.762} - 1} \approx \frac{1}{5.82 - 1} \approx 0.207
  $$
  Substantial microwave background exists!

- For optical light ($\nu = 500\text{ THz}$):
  $h\nu \approx 3.313 \times 10^{-19}\text{ J} \approx 2.07\text{ eV}$.
  Ratio $x = \frac{2.07\text{ eV}}{0.235\text{ meV}} \approx 8800$.
  Mean photon number:
  $$
  \bar{n} = \frac{1}{e^{8800} - 1} \approx e^{-8800} \approx 10^{-3822} \equiv 0
  $$

**Part 2 Solution:**
At room temperature ($T = 300\text{ K}$), $k_BT \approx 25.9\text{ meV}$. For visible photons ($h\nu \sim 2\text{ eV}$), $h\nu \gg k_BT$, so optical modes are in their absolute quantum ground state $|0\rangle$ with zero thermal background noise!
In contrast, for superconducting microwave qubits operating at $5\text{ GHz}$, $h\nu \approx 20\mu\text{eV} \ll k_B(300\text{ K})$. At room temperature, $\bar{n} \approx \frac{k_BT}{h\nu} \approx 1250$ photons! To reach the quantum ground state ($\bar{n} < 0.01$), superconducting qubits must be cooled below $20\text{ mK}$ in dilution refrigerators.
</details>

---

### Challenge 2: Numerical Verification with QuTiP
Use QuTiP's built-in `qutip.thermal_dm(N, n_bar)` to construct a thermal density matrix and verify that the trace distance $D(\rho_{\text{num}}, \rho_{\text{qutip}}) = \frac{1}{2}\mathrm{Tr}|\rho_{\text{num}} - \rho_{\text{qutip}}|$ is $< 10^{-7}$.

<details>
<summary><b>Click to expand Solution to Challenge 2</b></summary>

```python
if HAS_QUTIP:
    import qutip as qt
    N = 40
    n_bar = 3.2
    
    # QuTiP thermal state
    rho_qt = qt.thermal_dm(N, n_bar)
    
    # NumPy thermal state
    rho_np, _, _ = thermal_density_matrix(n_bar, N)
    
    # Trace distance
    diff = rho_qt.full() - rho_np
    trace_dist = 0.5 * np.sum(np.abs(la.eigvalsh(diff)))
    print(f"Trace distance between NumPy and QuTiP: {trace_dist:.2e}")
    assert trace_dist < 1e-7, "Trace distance benchmark failed!"
else:
    print("QuTiP not installed; run in Google Colab to execute this test.")
```
</details>


## 5. Summary, Reference Table, and Outlook

### Comprehensive Comparison: Single-Mode Light States

| Metric | Fock State $|n\rangle$ | Coherent State $|\alpha\rangle$ | Squeezed Vacuum $|0, r\rangle$ | Thermal State $\hat{\rho}_{\text{th}}$ |
| :--- | :--- | :--- | :--- | :--- |
| **State Type** | Pure ($|\psi\rangle$) | Pure ($|\psi\rangle$) | Pure ($|\psi\rangle$) | Mixed ($\hat{\rho}$) |
| **Purity $\mathrm{Tr}(\rho^2)$** | $1.0$ | $1.0$ | $1.0$ | $\frac{1}{2\bar{n}+1} < 1$ |
| **Entropy $S(\rho)$** | $0$ | $0$ | $0$ | $(\bar{n}+1)\ln(\bar{n}+1) - \bar{n}\ln\bar{n}$ |
| **Photon Statistics** | Deterministic: $\delta_{n, n_0}$ | Poissonian: $e^{-\bar{n}}\frac{\bar{n}^n}{n!}$ | Even-Fock: $P_{2m+1}=0$ | Bose-Einstein: $\frac{\bar{n}^n}{(1+\bar{n})^{n+1}}$ |
| **Variance $\text{Var}(n)$** | $0$ | $\bar{n}$ | $2\bar{n}(\bar{n}+1)$ | $\bar{n} + \bar{n}^2$ |
| **Mandel $Q$** | $-1$ (Sub-Poissonian) | $0$ (Poissonian) | $1 + 2\bar{n} > 0$ (Super) | $+\bar{n} > 0$ (Super) |
| **Phase Coherence** | None | Full | Quadrature-locked | None |

---

### Key Takeaways
1. **The Vacuum Dominance:** For any thermal state with finite temperature, $P_0 = 1/(1+\bar{n})$ is strictly the global maximum of $P_n$.
2. **Einstein's 1909 Fluctuation Formula:** Thermal photon noise decomposes cleanly into particle shot noise $\bar{n}$ and wave interference noise $\bar{n}^2$.
3. **Purity Decay:** State purity scales inversely with thermal photon occupancy $\mathcal{P} = 1/(2\bar{n}+1)$, providing an operational benchmark for decoherence.

---

### Primary References
- **Einstein, A.** (1909). *Über die Entwicklung unserer Anschauungen über das Wesen und die Konstitution der Strahlung*. Physikalische Zeitschrift, 10, 817–825.
- **Planck, M.** (1901). *Ueber das Gesetz der Energieverteilung im Normalspektrum*. Annalen der Physik, 309(3), 553–563.
- **Gerry, C. C., & Knight, P. L.** (2005). *Introductory Quantum Optics*. Cambridge University Press. Chapter 3: Coherence Properties of the Electromagnetic Field.
- **Mandel, L., & Wolf, E.** (1995). *Optical Coherence and Quantum Optics*. Cambridge University Press. Chapter 13: Thermal Fields.

---

### Next in the Curriculum
In **Notebook 1.6: The Mandel Q-Parameter and Non-Classicality Criteria**, we formalize the boundary between classical and quantum light, comparing Fano factors, sub-Poissonian counting, and non-classicality witnesses across all state families.
